**数据预处理：pandas**<a id='toc0_'></a>    
- [读取数据集](#toc1_)    
- [处理缺失值](#toc2_)    
- [转换为张量格式](#toc3_)    
- [小结](#toc4_)    
- [练习](#toc5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<div style="font-size:32px;  padding:5px;"> 
数据预处理：pandas
</div>

为了能用深度学习来解决现实世界的问题，我们经常从预处理原始数据开始，
而不是从那些准备好的张量格式数据开始。
在Python中常用的数据分析工具中，我们通常使用[pandas库](https://pandas.pydata.org/docs/user_guide/index.html)软件包。
像庞大的Python生态系统中的许多其他扩展包一样，`pandas`可以与张量兼容。
本节内容虽然不能替代一份系统的 *pandas*
[教程](https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html)，
但将为你提供一次速成入门，
介绍一些最常见的操作流程。
后面的章节将介绍更多的数据预处理技术。

⚠️ `pandas`中`fillna()`等数据处理方法默认是非原地（non-inplace）的

$\implies$ 它们不会直接修改原始`DataFrame/Series`，而是返回一个修改后的新副本。

# <a id='toc1_'></a>[读取数据集](#toc0_)

逗号分隔值（CSV）文件在存储表格型（类似电子表格）数据时极为常见。
在这种文件中，每一行对应一条记录，
并由若干个（以逗号分隔的）字段组成，例如：
> “Albert Einstein,March 14 1879,Ulm,Federal polytechnic school,field of gravitational physics”.

为了演示如何使用 `pandas` 加载 CSV 文件，我们首先**创建一个人工数据集，并存储在CSV（逗号分隔值）文件**
`../data/house_tiny.csv`中。

该文件表示一个房屋数据集，其中:
* 每一行对应一套不同的房屋，
* 各列分别表示房间数量（`NumRooms`）、屋顶类型（`RoofType`）以及价格（`Price`）。

以其他格式存储的数据也可以通过类似的方式进行处理。

In [1]:
import os
# 手搓数据集
os.makedirs(os.path.join('..', 'data'), exist_ok=True)# 如果该文件夹存在的话，也不报错而是继续运行
data_file = os.path.join('..', 'data', 'house_tiny.csv')# 如果这csv存在，也不会报错，因为只是拼接路径
with open(data_file, 'w') as f:# 如果不存在就创建+写入，已存在就清空+写入，不想清空就用a
    f.write('''NumRooms,RoofType,Price
NA,NA,127500
2,NA,106000
4,Slate,178100
NA,NA,140000''')#第一行是列名

🔔 【数据读取】操作如下。我们导入`pandas`包并调用`read_csv`函数。该数据集有四行三列。

In [1]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
# !pip install pandas
import os
import pandas as pd # type: ignore
data_file = os.path.join('..', 'data', 'house_tiny.csv')
data = pd.read_csv(data_file)
print(data)

   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


请注意，此时`pandas` 已将CSV 中所有取值为 `NA` 的条目
替换为一种 `NaN`（*not a number*而不是例如9999这样的具体的数）值。

当某个条目为空时也会发生这种替换情况，例如 `"3,,,270000"`。
这些被称为*缺失值*。它们是数据科学中的“臭虫”，是一种在你整个职业生涯中都会反复遇到的顽固问题。

# <a id='toc2_'></a>[缺失值处理`get_dummies;fillna`](#toc0_)

在监督学习中，我们训练模型在给定一组*输入*值的情况下，去预测一个指定的*目标*值。

1️⃣ 处理数据集的第一步是将对应于输入值和目标值的列分离开来。

🔔 我们既可以按【列名】选择列（`loc`：Slicing is inclusive on both ends），也可以通过基于【整数位置】的索引（`iloc`：Slicing is left-closed, right-open (like Python)）来选择列。

In [6]:
d1 = data.loc[3, 'Price']                 # row label 3, column 'Price'
d2 = data.loc[:, 'NumRooms':'Price']        # includes from 'NumRooms' to 'Price'
d3 = data[['RoofType','Price']]

print(d1)
print("*********")
print(d2)
print("*********")
print(d3)

140000
*********
   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000
*********
  RoofType   Price
0      NaN  127500
1      NaN  106000
2    Slate  178100
3      NaN  140000


In [ ]:
inputs, targets = data.iloc[:, 0:2], data.iloc[:, 2]# iloc的切片不会包括列表的最后一个，但是loc的会把列表的每项都选中

print("inputs:")# DataFrame (二维表格)
print(inputs)

print("targets:")# Series (一维数组)
print(targets)# 当你使用整数索引 iloc[:, 2] 取出单列时，Pandas 会默认将数据“降维”。它不再是一个表格，而变成了一个带有标签的一维数组
# 在打印一维数组的时候，是先打印数据（索引和值），再在底部打印该序列的元数据（metadata），包括那么name:原本列名，dtype：数据类型

inputs:
   NumRooms RoofType
0       NaN      NaN
1       2.0      NaN
2       4.0    Slate
3       NaN      NaN
targets:
0    127500
1    106000
2    178100
3    140000
Name: Price, dtype: int64


In [ ]:
# 补充：保持一维数组也是dataframe格式
# 思路就是要求：取出来的这一列，也是使用列表索引出来的

# 方法1
# 注意这里的 [2]，这告诉 pandas "我要取出第2列组成的列表"，而不是"取出第2列"
targets1 = data.iloc[:, [2]]
print(targets1)

# 方法2
# 表示从第2列切到第3列（不含），结果仍然是 DataFrame
targets2 = data.iloc[:, 2:3]
print(targets2)

使用`iloc`索引找列有个小小的问题：当你在表里新插入一列的时候，索引没变，那找到的可能不是你想要的玩意儿。

2️⃣ 现在开始处理缺失值。常见处理方法如下，请结合具体情况使用：

* 【**插值法mputation**】：这个缺失值的估计值是啥，再用估计值替换。
* 【**删除法deletion**】：直接丢掉包含缺失值的行或者列。

这个例子里我们考虑插值法。离散和连续的两大类数据，都有对应的启发式插值法：

🌸 **对于【分类型】输入字段，我们可以将 `NaN` 视为一个类别。**

由于 `RoofType` 列的取值为 `Slate` 和 `NaN`，因此`pandas` 可以将其
转换为两列：`RoofType_Slate` 和 `RoofType_nan`。

$\implies$ 如果某一行的屋顶类型是 `Slate`，
那么 `RoofType_Slate` 和 `RoofType_nan`
的取值分别为 1 和 0。
反之，对于屋顶类型缺失的那一行，
取值成了 0 和 1。

In [ ]:

inputs = pd.get_dummies(inputs, dummy_na=True)# # 对inputs中的分类特征执行独热编码（将离散类别转为数值型特征）  
# dummy_na=True → 特殊处理：将缺失值(NA)视为一个独立的类别，生成对应的编码列  
# 例如：若某列有"NA"值，会新增一列如"列名_na"来标识该缺失情况 
print(inputs)

In [4]:
# 也可以使用这句
inputs = pd.get_dummies(inputs, dummy_na=True, dtype=int)#不设置为int，独热编码出现的就是false或者true
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       NaN               0             1
1       2.0               0             1
2       4.0               1             0
3       NaN               0             1


🌸 **对于【数值型】数据，缺失值可以用对应【列的平均值】来替换这些 `NaN` 项。**

In [5]:
inputs = inputs.fillna(inputs.mean())
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       3.0               0             1
1       2.0               0             1
2       4.0               1             0
3       3.0               0             1


⚠️ 请注意`pandas`使用`[]`会被误解成按列名筛选，因此需要使用位置索引`iloc`或者标签索引`loc`来选择

In [ ]:
inputs['NumRooms'] = inputs['NumRooms'].fillna(inputs['NumRooms'].mean())# 当然也可以直接使用列名筛选
print(inputs)

# <a id='toc3_'></a>[转换为张量格式](#toc0_)

现在`inputs`和`outputs`中的所有条目都是【**数值类型** 】

$\implies$ 可以转换为【**张量格式**】。

当数据采用张量格式后，可以通过在[ndarray](ndarray.ipynb)中引入的那些张量函数来进一步操作。


In [6]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(targets.to_numpy(dtype=float))
X, y

(tensor([[3., 0., 1.],
         [2., 0., 1.],
         [4., 1., 0.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

# <a id='toc4_'></a>[小结](#toc0_)

你现在已经学会了如何划分数据列、
对缺失变量进行填补，
以及将 pandas 数据加载为张量。
在后面的kaggle练习中，
你还将掌握更多的数据处理技巧。

虽然这门速成课程刻意保持了内容的简洁，
但在实际应用中，数据处理往往会变得相当复杂。
例如，数据集可能并不是以单个 CSV 文件的形式提供，
而是分散在从关系型数据库中提取的多个文件里。
在电子商务应用中，
客户地址可能存储在一张表中，
而购买记录则存储在另一张表中。

此外，实际工作中还会遇到
远不止类别型和数值型的数据类型，
例如文本字符串、图像、
音频数据以及点云等。
很多时候，为了避免数据处理成为机器学习流水线中最大的瓶颈，
必须借助高级工具和高效算法。
当我们学习计算机视觉和自然语言处理时，
这些问题就会变得尤为突出。

最后，我们还必须重视数据质量。
真实世界的数据集往往充满了
异常值、传感器的错误测量以及记录错误，
在将数据输入任何模型之前，
都必须先妥善处理这些问题。
诸如 [seaborn](https://seaborn.pydata.org/)，
[Bokeh](https://docs.bokeh.org/)
或[matplotlib](https://matplotlib.org/)
等数据可视化工具，
可以帮助你手动检查数据，
并逐步建立起对
所需解决问题类型的直觉理解。

# <a id='toc5_'></a>[练习](#toc0_)

创建包含更多行和列的原始数据集。

1. 删除缺失值最多的列。
2. 将预处理后的数据集转换为张量格式。

In [ ]:
import os
import pandas as pd
data_file = os.path.join('..', 'data', 'house_tiny.csv')
data = pd.read_csv(data_file)


In [ ]:
# 删除缺失值最多的列
null_count = data.iloc[:,0:2].isnull().sum()
max_null = null_count.idxmax()
data.drop(max_null, axis = 1, inplace=True)
print(data)

In [ ]:
data_tensor = torch.tensor(data.values)
print(data_tensor)

附加题：

1. 尝试加载数据集，例如从[UCI机器学习仓库](https://archive.ics.uci.edu/ml/datasets.php)获取的鲍鱼（Abalone）数据集，并检查它们的属性。其中有缺失值的数据集占比多少？变量中数值型、类别型或文本型的占比分别是多少？

2. 尝试通过列名而非列号来索引和选择数据列。pandas文档中关于[索引](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html) 的部分有更详细的操作说明。

3. 你认为通过这种方式可以加载多大的数据集？可能存在哪些限制因素？提示：考虑数据读取时间、数据表示方式、处理过程和内存占用。在你的笔记本电脑上尝试一下，若在服务器上尝试会发生什么？

4. 如何处理具有大量类别的数据？如果类别标签全都是唯一的呢？是否应该保留后者？

5. 你能想到哪些pandas的替代工具？比如[从文件中加载NumPy张量？](https://numpy.org/doc/stable/reference/generated/numpy.load.html)? 可以了解一下[Pillow](https://python-pillow.org/)——Python图像处理库。

In [ ]:
#!pip install ucimlrepo
# ucimlrepo is a parser, not a file mirror

这个是安装uci解释器，用于读取数据data+文档names，并转为python能使用的结构对象。也就是哪怕uci网站显示数据文件包括：
* abalone.data
* abalone.names
* Index

ucimlrepo也会提取成python友好结构，如下图
```kotlin
UCI folder
├── abalone.data (parsed)    → actual data: features or input variables (X) + target variable (y)
├── abalone.names(parsed&discarded)    → documentation for humanbeings
│   │ 
│   ├── dataset description ──▶ abalone.metadata(dataset-level info)
│   │
│   └── attribute list ───────▶ abalone.variables(variable-level info)
│
└── Index (discarded)        → file listing (HTML, no semantic value)
```


要查看names内容，需要下载文件或者直接去[网页](https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.names)访问，不能使用解析器。

而index文件不是数据集的内容，而只是uci网站的目录列表文件。ucimlrepo解析器是设计出来为你提供数据+元数据，而非提供网页产物，所以把这个废弃了。

In [7]:
# https://archive.ics.uci.edu/dataset/1/abalone
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 

In [8]:
abalone.metadata

{'uci_id': 1,
 'name': 'Abalone',
 'repository_url': 'https://archive.ics.uci.edu/dataset/1/abalone',
 'data_url': 'https://archive.ics.uci.edu/static/public/1/data.csv',
 'abstract': 'Predict the age of abalone from physical measurements',
 'area': 'Biology',
 'tasks': ['Classification', 'Regression'],
 'characteristics': ['Tabular'],
 'num_instances': 4177,
 'num_features': 8,
 'feature_types': ['Categorical', 'Integer', 'Real'],
 'demographics': [],
 'target_col': ['Rings'],
 'index_col': None,
 'has_missing_values': 'no',
 'missing_values_symbol': None,
 'year_of_dataset_creation': 1994,
 'last_updated': 'Mon Aug 28 2023',
 'dataset_doi': '10.24432/C55C7W',
 'creators': ['Warwick Nash',
  'Tracy Sellers',
  'Simon Talbot',
  'Andrew Cawthorn',
  'Wes Ford'],
 'intro_paper': None,
 'additional_info': {'summary': 'Predicting the age of abalone from physical measurements.  The age of abalone is determined by cutting the shell through the cone, staining it, and counting the number of r

In [ ]:
abalone.variables# 在这里就可以看到习题的答案了

,name,role,type,demographic,description,units,missing_values
0,Sex,Feature,Categorical,None,"M, F, and I (infant)",None,no
1,Length,Feature,Continuous,None,Longest shell measurement,mm,no
2,Diameter,Feature,Continuous,None,perpendicular to length,mm,no
3,Height,Feature,Continuous,None,with meat in shell,mm,no
4,Whole_weight,Feature,Continuous,None,whole abalone,grams,no
5,Shucked_weight,Feature,Continuous,None,weight of meat,grams,no
6,Viscera_weight,Feature,Continuous,None,gut weight (after bleeding),grams,no
7,Shell_weight,Feature,Continuous,None,after being dried,grams,no
8,Rings,Target,Integer,None,+1.5 gives the age in years,None,no


In [14]:
# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets 
  
# metadata 
print(X.head()) 

print("***********")
# variable information 
print(y.head()) 


  Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  
0         0.150  
1         0.070  
2         0.210  
3         0.155  
4         0.055  
***********
   Rings
0     15
1      7
2      9
3     10
4      7
